[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/state_space/state_space.ipynb)

# Heat Capacity Under a Constraint

This example computes how much a gas warms when it takes in heat under a constraint. A gas in a cylinder with a latched piston keeps its volume fixed, so all added heat raises its internal energy and warms it; under a piston loaded with a weight it keeps its pressure fixed instead, and part of the heat goes into lifting the weight, so the same heat warms it less. The heat required per unit rise in temperature, its heat capacity, depends on which condition holds along the path.

A state of the gas is modelled as an antivector in the exterior plane `s0 v0` of entropy `s.dual()` and volume `v.dual()`, with each extensive variable read by pairing (`s & state` and `v & state`). An ideal gas's internal energy sets its conjugate potentials, temperature along `s` and minus pressure along `v`, together with the covectors that pair with a step in state through `&` to read how temperature and pressure change. A constraint is another covector whose pairing `constraint & step` vanishes along an allowed path. Dividing `constraint` by `(temperature_slope ^ constraint).dual()` gives the change of state per unit rise in temperature for any linear constraint, without coordinates or matrix solves.

In thermodynamic partial-derivative notation, where subscripts track which variable is held fixed, stepping per unit $T$ along constant $g(S, V)$ takes the Jacobian ratio $\left(\frac{\partial S}{\partial T}\right)_g = \frac{(\partial g/\partial V)_S}{(\partial T/\partial S)_V(\partial g/\partial V)_S - (\partial T/\partial V)_S(\partial g/\partial S)_V}$, and relating $C_p = T(\partial S/\partial T)_p$ to $C_V = T(\partial S/\partial T)_V$ invokes $\left(\frac{\partial p}{\partial T}\right)_V \left(\frac{\partial T}{\partial V}\right)_p \left(\frac{\partial V}{\partial p}\right)_T = -1$ to obtain $C_p - C_V = -T\left(\frac{\partial p}{\partial T}\right)_V^2 \!\big/\! \left(\frac{\partial p}{\partial V}\right)_T$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import show_states
from examples.thermodynamics.state_space import render

# Exterior plane of entropy (in units of nR) and volume (in units of V₀); temperature is in T₀ and energy in nRT₀.
ga = Algebra("s0v0")
mv = NumpyContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
State = ga.gatype.antivector()                                                # Antivector-
Potential = ga.gatype.vector()                                                # Vector+

thermal_capacity = 1.5
n_cylinders = 2

## Temperature and pressure from the energy

The ideal gas's internal energy grows exponentially with entropy and falls as a power of volume. Differentiating along entropy and volume gives the temperature and the pressure, while differentiating a second time gives the covector slopes `temperature_slope` and `pressure_slope` that pair with any change of state through `&` to read how temperature and pressure change. The temperature–entropy chart below plots isochores in green and isobars in purple, while the pressure–volume chart plots isotherms in red and adiabats in teal, with both cylinders starting at the marked reference state.

In [ ]:
def energy(state: State) -> Scalar:
    """Internal energy of an ideal gas with a constant heat capacity at fixed volume."""
    entropies = mv.s & state                                                  # [...] Scalar+
    volumes = mv.v & state                                                    # [...] Scalar+
    return thermal_capacity * ((entropies - volumes.log()) / thermal_capacity).exp()


def response(state: State) -> tuple[Scalar, Scalar, Potential, Potential]:
    """Temperature, pressure, and the potential slopes pairing with a step in state to give their changes."""
    volumes = mv.v & state                                                    # [...] Scalar+
    temperatures = energy(state) / thermal_capacity                           # [...] Scalar+
    pressures = temperatures / volumes                                        # [...] Scalar+
    temperature_slope = (temperatures / thermal_capacity) * (mv.s - mv.v / volumes)                # [...] Potential+
    pressure_slope = ((pressures / thermal_capacity) * mv.s
                      - (pressures * (1 + 1 / thermal_capacity) / volumes) * mv.v)                 # [...] Potential+
    return temperatures, pressures, temperature_slope, pressure_slope


def temperature_of(states: State) -> Scalar:
    """Temperatures of the gas states."""
    return response(states)[0]


def pressure_of(states: State) -> Scalar:
    """Pressures of the gas states."""
    return response(states)[1]


# Initialize both cylinders at unit volume, unit temperature, and zero reference entropy.
start = (mv.s.dual() * 0 + mv.v.dual())[None].broadcast_to((n_cylinders,))    # [n_cylinders] State-
render.draw_charts(temperature_of, pressure_of, start);

## The heating direction

The latched cylinder fixes its volume through the covector `v`, while the loaded cylinder fixes its pressure through `pressure_slope`. In the exterior plane `s0 v0`, dividing `constraint` by the intrinsic scalar `(temperature_slope ^ constraint).dual()` turns the constraint covector into the state step that pairs to zero with `constraint` and to one with `temperature_slope`; multiplying the temperature by `s & direction` then gives the heat capacity under that constraint. On the charts, the arrows follow the isochore for fixed volume and the isobar for fixed pressure.

In [ ]:
def heating_direction(state: State, constraint: Potential) -> tuple[State, Scalar]:
    """Change of state per unit rise in temperature along `constraint & step == 0`, and the heat capacity along it."""
    temperatures, _, temperature_slope, _ = response(state)
    # Dividing the constraint covector by the intrinsic scalar `(temperature_slope ^ constraint).dual()` gives the
    # intrinsic state step that pairs to zero with the constraint and to one with the temperature slope.
    direction = constraint / (temperature_slope ^ constraint).dual()          # [...] State-
    return direction, temperatures * (mv.s & direction)


def constraints(states: State) -> Potential:
    """Covectors paired to zero along each cylinder's path, fixing volume on the latched cylinder and pressure on the
    loaded one."""
    _, _, _, pressure_slope = response(states)
    return stack((mv.v, pressure_slope[1]))                                   # [n_cylinders] Potential+


directions, heat_capacities = heating_direction(start, constraints(start))
render.draw_directions(temperature_of, pressure_of, start, directions);

In [ ]:
print("heat capacities, latched and loaded:", heat_capacities.to_array())

## Equal heat

Both cylinders receive equal slices of heat, advancing each state along its midpoint heating direction by the added heat divided by its heat capacity. Reversible heating adds heat equal to the area under the path on the temperature–entropy chart down to zero temperature, so both shaded areas grow equally, while the area under the pressure–volume path measures the mechanical work done by the loaded piston. The third chart plots each cylinder's temperature against the cumulative heat added as straight lines whose slopes equal the reciprocal heat capacities.

This cell takes about ten seconds, most of it drawing the animation.

In [ ]:
total_heat, n_steps, frame_duration_ms = 2.4, 97, 40


def heating_step(states: State, heat: float) -> State:
    """Cylinder states after adding a slice of heat along their constrained heating directions by the midpoint rule."""
    direction, capacity = heating_direction(states, constraints(states))
    halfway = states + direction * (heat / capacity / 2)                      # [n_cylinders] State-
    direction, capacity = heating_direction(halfway, constraints(halfway))
    return states + direction * (heat / capacity)                             # [n_cylinders] State-


def equal_heating():
    """Yields the latched and loaded cylinders' states, their heating directions, and the cumulative heat added at
    each step."""
    states = start
    for step in range(n_steps + 1):
        directions, _ = heating_direction(states, constraints(states))
        yield states, directions, step * total_heat / n_steps
        states = heating_step(states, total_heat / n_steps)


stages = show_states(equal_heating(), render.heating, temperature_of, pressure_of, name="thermodynamics_heating",
                     duration_ms=frame_duration_ms, n_frames=n_steps + 1)

The latched gas in blue warms along its isochore and does no work, while the loaded gas in orange warms more slowly along its isobar and expands to lift the weight. Because both heat capacities are constant for this ideal gas (`1.5` and `2.5`), adding `2.4` units of heat raises the temperatures from `1` to `2.6` and `1.96`.

In [ ]:
end = stages[-1][0]                                                           # [n_cylinders] State-
final_temperatures = temperature_of(end)                                      # [n_cylinders] Scalar+
print("final temperatures, latched and loaded:", final_temperatures.to_array())

In [ ]:
# checks
np.testing.assert_allclose(heat_capacities.to_array(), [thermal_capacity, thermal_capacity + 1], atol=1e-10)
np.testing.assert_allclose(final_temperatures.to_array(), [2.6, 1.96], rtol=1e-3)
# Energy gained plus work done against the fixed load equals the total heat added.
work = pressure_of(start) * ((mv.v & end) - (mv.v & start))
np.testing.assert_allclose((energy(end) - energy(start) + work).to_array(), total_heat, rtol=1e-3)
np.testing.assert_allclose((mv.v & end[0]).to_array(), 1, atol=1e-10)